# Cattle keypoint detection

A standalone Python **3.11** notebook using PyTorch and torchvision, without the reference repository's legacy MMPose/MMCV environment. Run cells in order.

**Scope:** 18-point top-down pose estimation given a cattle bounding box. Evaluation uses ground-truth boxes, so these results measure localization, not full-image cattle detection. ImageNet pretraining supplies generic features; cattle supervision comes exclusively from the reference dataset.

**Attribution:** Wang et al., *Performance evaluation of a state-of-the-art keypoint detection method for precision livestock farming*, Computers and Electronics in Agriculture 240 (2026), 111230. [Repository](https://github.com/yww0411/Livestock-keypoint-detection), [paper](https://doi.org/10.1016/j.compag.2025.111230). Dataset: CC BY 4.0; reference code: MIT. Data downloads are commit-pinned and retain the license.

## 1. Environment and configuration

Use the existing Python 3.11 kernel. No automatic package upgrades. Tested: Python 3.11.4, torch 2.2.0+cu121, torchvision 0.17.0+cu121, NumPy 1.26.4. Also uses installed Pillow, OpenCV, pandas, matplotlib, tqdm and Jupyter.

Default: 100 epochs, validation early stopping. Pause/resume and monitoring instructions appear before the training cell. Keep one trainer per run directory.

In [ ]:
from pathlib import Path
import sys, os, json, random, hashlib, time, platform
from collections import Counter
import numpy as np
import pandas as pd
# Load the native renderer before torch/OpenCV to avoid a Windows DLL import-order crash.
from matplotlib.backends.backend_agg import FigureCanvasAgg
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from tqdm.auto import tqdm
from IPython.display import display

ROOT=Path.cwd().resolve()
if ROOT.name=='ml': ROOT=ROOT.parent
assert (ROOT/'ml').is_dir(), 'Open from the repository root or ml directory.'
CFG=dict(seed=3407,fold='p1A',width=256,height=192,batch_size=16,epochs=100,
         patience=20,lr=.0003,weight_decay=.0001,freeze_epochs=4,sigma=1.5,crop_padding=1.25,pretrained=True)
COMMIT='d2fbc81e6181751f2e65eb6e393b10a9c7e16dfe'
DATA=ROOT/'ml/data/reference'; CATTLE=DATA/'data_process/cattle'
RUN=ROOT/'ml/runs/cattle_resnet18_p1A'; RUN.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
if torch.cuda.is_available(): torch.cuda.manual_seed_all(CFG['seed'])
torch.set_num_threads(min(4,os.cpu_count() or 1))
torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
ENV=dict(python=sys.version,torch=torch.__version__,device=str(DEVICE),numpy=np.__version__,
         gpu=torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None,platform=platform.platform())
print(json.dumps(ENV,indent=2)); print('Run folder:',RUN)
assert sys.version_info>=(3,10)

## 2. Import the reference data

Download only cattle data, attribution and the license (about 133 MB). All files are verified with their Git blob hashes. Downloads use temporary files and retry transient errors. No upstream training code is executed.

In [ ]:
import urllib.request
from concurrent.futures import ThreadPoolExecutor
BASE=f'https://raw.githubusercontent.com/yww0411/Livestock-keypoint-detection/{COMMIT}/'
def get_bytes(url):
    for attempt in range(4):
        try:
            with urllib.request.urlopen(url,timeout=90) as response: return response.read()
        except Exception:
            if attempt==3: raise
            time.sleep(2**attempt)
manifest_path=DATA/'manifest.json'
if manifest_path.exists():
    manifest=json.loads(manifest_path.read_text()); assert manifest['sha']==COMMIT
else:
    manifest=json.loads(get_bytes(f'https://api.github.com/repos/yww0411/Livestock-keypoint-detection/git/trees/{COMMIT}?recursive=1'))
    assert not manifest.get('truncated')
    DATA.mkdir(parents=True,exist_ok=True); manifest_path.write_text(json.dumps(manifest))
files=[x for x in manifest['tree'] if x['type']=='blob' and
       (x['path'].startswith('data_process/cattle/') or x['path'] in ('README.md','LICENSE-DATA.txt'))]
def git_hash(b): return hashlib.sha1(f'blob {len(b)}\0'.encode()+b).hexdigest()
def download_one(item):
    path=DATA/item['path']
    if path.exists() and git_hash(path.read_bytes())==item['sha']: return
    content=get_bytes(BASE+item['path']); assert git_hash(content)==item['sha'], f'Integrity failure: {path}'
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.part'); tmp.write_bytes(content); tmp.replace(path)
with ThreadPoolExecutor(max_workers=8) as pool:
    list(tqdm(pool.map(download_one,files),total=len(files),desc='Verify/download cattle data'))
provenance=dict(repository='https://github.com/yww0411/Livestock-keypoint-detection',commit=COMMIT,dataset_license='CC BY 4.0',files=len(files))
(DATA/'source.json').write_text(json.dumps(provenance,indent=2))
print((DATA/'LICENSE-DATA.txt').read_text())

## 3. Validate data and train/validation/test separation

Use the published **p1A** protocol: train/validation use source groups B/C; test uses held-out group A. Validate decoded image dimensions, annotation IDs, cattle categories, keypoint names, boxes, coordinates and visibility. Invalid input raises an error.

The published split has nine original filename groups shared by train and validation. Conservatively move training members of these groups into validation; leave test unchanged. Filenames can collide across sources, so this is a precaution, not proof of duplicate content. Exact hashes and original filenames must then be disjoint. Save the actual split manifest and fingerprint. These checks cannot exclude different frames of the same animal/video because upstream recording/animal identities are unavailable. This is a source-group holdout, not a verified animal-independent split. Test labels are not used for training or model selection.

In [ ]:
NAMES=['mouth','eye','ear','neck','shoulder','chest','hip','tail','elbow','l_fore_wrist',
       'l_fore_foot','r_fore_wrist','r_fore_foot','hind_knee','l_hind_hock','l_hind_foot','r_hind_hock','r_hind_foot']
K=len(NAMES); records={}; audit=[]
for split in ('train','val','test'):
    coco=json.loads((CATTLE/'annotations_p1'/f"cattle_{CFG['fold']}_{split}.json").read_text())
    images={i['id']:i for i in coco['images']}
    assert len(images)==len(coco['images'])
    assert len({a['id'] for a in coco['annotations']})==len(coco['annotations'])
    cats={c['id']:c for c in coco['categories']}; rows=[]
    for a in coco['annotations']:
        assert a['image_id'] in images
        assert cats[a['category_id']]['name'].lower()=='cattle'
        assert cats[a['category_id']]['keypoints']==NAMES
        im=images[a['image_id']]; path=(CATTLE/im['file_name']).resolve()
        assert path.is_relative_to(CATTLE.resolve())
        with Image.open(path) as pil:
            pil.load(); assert pil.size==(im['width'],im['height'])
        kp=np.asarray(a['keypoints'],dtype=np.float32).reshape(K,3)
        box=np.asarray(a['bbox'],dtype=np.float32)
        assert np.isfinite(kp).all() and np.isfinite(box).all()
        assert set(kp[:,2]).issubset({0.,1.,2.})
        m=kp[:,2]>0; assert m.any() and int(m.sum())==a['num_keypoints']
        assert (kp[m,:2]>=0).all()
        # COCO coordinates are continuous; points on width/height boundaries are valid.
        assert (kp[m,0]<=im['width']).all() and (kp[m,1]<=im['height']).all()
        x,y,w,h=box
        assert x>=0 and y>=0 and w>0 and h>0
        assert x+w<=im['width']+1 and y+h<=im['height']+1
        original=im.get('extra',{}).get('name',Path(im['file_name']).name.split('.rf.')[0])
        rows.append(dict(image_id=im['id'],annotation_id=a['id'],path=str(path),file_name=im['file_name'],
                         group=im['batch_id'],original=original.lower(),sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                         bbox=box.tolist(),keypoints=kp.tolist()))
    records[split]=rows
    audit.append(dict(split=split,images=len(images),animals=len(rows),
                      labeled=sum(np.count_nonzero(np.array(r['keypoints'])[:,2]) for r in rows),
                      visible=sum(np.count_nonzero(np.array(r['keypoints'])[:,2]==2) for r in rows),
                      groups=dict(Counter(r['group'] for r in rows))))
colliding={r['original'] for r in records['train']} & {r['original'] for r in records['val']}
moved=[r for r in records['train'] if r['original'] in colliding]
records['train']=[r for r in records['train'] if r['original'] not in colliding]
records['val'].extend(moved)
print('Conservative split repair:',len(colliding),'filename groups;',len(moved),'animals moved train -> validation')
audit=[]
for split,rows in records.items():
    audit.append(dict(split=split,images=len({r['image_id'] for r in rows}),animals=len(rows),
                      labeled=sum(np.count_nonzero(np.array(r['keypoints'])[:,2]) for r in rows),
                      visible=sum(np.count_nonzero(np.array(r['keypoints'])[:,2]==2) for r in rows),
                      groups=dict(Counter(r['group'] for r in rows))))
for i,a in enumerate(('train','val','test')):
    for b in ('train','val','test')[i+1:]:
        for key in ('sha256','original'):
            assert not ({r[key] for r in records[a]} & {r[key] for r in records[b]}), f'{key} leakage: {a}/{b}'
assert {r['group'] for r in records['test']}=={'A'}
assert not ({r['group'] for r in records['train']+records['val']} & {'A'})
split_manifest={s:[{k:v for k,v in r.items() if k not in ('path','keypoints')} for r in rr] for s,rr in records.items()}
SPLIT_HASH=hashlib.sha256(json.dumps(split_manifest,sort_keys=True).encode()).hexdigest()
(RUN/'split_manifest.json').write_text(json.dumps(split_manifest,indent=2))
pd.DataFrame(audit).to_csv(RUN/'data_audit.csv',index=False)
display(pd.DataFrame(audit)); print('Split fingerprint:',SPLIT_HASH)

## 4. Inspect labels and prepare crops

Expand boxes with aspect-preserving padding and map to 256×192 pixels. Cache base crops in RAM. Training-only augmentation applies rotation, translation, scale, brightness and contrast. No horizontal flip: anatomical side and single eye/ear conventions are not fully specified. Visibility 0 is masked; visibility 1 (occluded) and 2 (visible) both supervise training. Augmented points outside the crop are masked for that sample.

In [ ]:
W,H=CFG['width'],CFG['height']; HW,HH=W//4,H//4
MEAN=np.array([.485,.456,.406],dtype=np.float32); STD=np.array([.229,.224,.225],dtype=np.float32)
def crop_geometry(box):
    x,y,w,h=box; cw=max(w,h*W/H)*CFG['crop_padding']; ch=cw*H/W
    return np.array([x+w/2-cw/2,y+h/2-ch/2,cw,ch],dtype=np.float32)
def make_crop(row):
    im=cv2.cvtColor(cv2.imread(row['path']),cv2.COLOR_BGR2RGB)
    x,y,cw,ch=crop_geometry(row['bbox'])
    matrix=np.array([[W/cw,0,-x*W/cw],[0,H/ch,-y*H/ch]],dtype=np.float32)
    crop=cv2.warpAffine(im,matrix,(W,H),flags=cv2.INTER_LINEAR,borderValue=(124,116,104))
    kp=np.array(row['keypoints'],dtype=np.float32); kp[:,:2]=kp[:,:2]@matrix[:,:2].T+matrix[:,2]
    return crop,kp
class CattleDataset(Dataset):
    def __init__(self,rows,augment=False):
        self.rows=rows; self.augment=augment; self.epoch=0
        self.cache=[make_crop(r) for r in tqdm(rows,desc='Cache crops')]
    def __len__(self): return len(self.rows)
    def __getitem__(self,index):
        image,kp=self.cache[index]; image=image.copy(); kp=kp.copy()
        if self.augment:
            rng=np.random.default_rng(CFG['seed']+self.epoch*100000+index)
            matrix=cv2.getRotationMatrix2D((W/2,H/2),float(rng.uniform(-20,20)),float(rng.uniform(.85,1.15))).astype(np.float32)
            matrix[:,2]+=rng.uniform(-.04,.04,2)*[W,H]
            image=cv2.warpAffine(image,matrix,(W,H),borderValue=(124,116,104))
            kp[:,:2]=kp[:,:2]@matrix[:,:2].T+matrix[:,2]
            image=np.clip(image.astype(np.float32)*rng.uniform(.8,1.2)+rng.uniform(-12,12),0,255)
        mask=(kp[:,2]>0)&(kp[:,0]>=0)&(kp[:,0]<W)&(kp[:,1]>=0)&(kp[:,1]<H)
        tensor=torch.from_numpy(((image.astype(np.float32)/255-MEAN)/STD).transpose(2,0,1).copy())
        return tensor,torch.from_numpy(kp[:,:2].copy()),torch.from_numpy(mask.astype(np.float32)),index
train_ds=CattleDataset(records['train'],augment=True); val_ds=CattleDataset(records['val'])
# Test crops are constructed only after checkpoint selection.
fig,axes=plt.subplots(2,3,figsize=(15,7))
for ax,(image,kp) in zip(axes.flat,train_ds.cache[:6]):
    ax.imshow(image); m=kp[:,2]>0
    ax.scatter(kp[m,0],kp[m,1],c=np.arange(K)[m],cmap='tab20',s=18)
    for j in np.flatnonzero(m): ax.text(kp[j,0],kp[j,1],str(j+1),fontsize=7,color='white')
    ax.axis('off')
plt.tight_layout(); plt.show()
display(pd.DataFrame({'index':np.arange(1,K+1),'keypoint':NAMES}))

## 5. Model, supervision and metrics

ResNet-18 starts with ImageNet weights; three learned upsampling stages output 18 independent 64×48 heatmaps. Spatial cross-entropy matches Gaussian target distributions without overwhelming keypoint foreground with background pixels. Warm up the decoder for four epochs, then fine-tune the backbone at one tenth the decoder learning rate. Backbone batch statistics stay frozen during warm-up.

Select checkpoints by **validation PCK@0.05**: Euclidean joint error divided by the original box diagonal. Also report PCK@0.10, mean normalized error (NME), and normalized area under the PCK curve on [0, 0.20]. Higher PCK/AUC and lower NME are better. These metrics are not OKS AP and must not be compared numerically with the paper's AP tables. Heatmap peaks are not calibrated visibility probabilities.

In [ ]:
class CattlePose(nn.Module):
    def __init__(self,pretrained=True):
        super().__init__()
        net=resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.backbone=nn.Sequential(*list(net.children())[:-2]); layers=[]
        for cin,cout in [(512,128),(128,64),(64,32)]:
            layers.extend([nn.ConvTranspose2d(cin,cout,4,2,1,bias=False),nn.BatchNorm2d(cout),nn.ReLU(inplace=True)])
        self.decoder=nn.Sequential(*layers,nn.Conv2d(32,K,1))
    def forward(self,x): return self.decoder(self.backbone(x))
def heatmap_loss(logits,points,mask):
    yy,xx=torch.meshgrid(torch.arange(HH,device=logits.device),torch.arange(HW,device=logits.device),indexing='ij')
    center=(points+.5)/4-.5
    target=torch.exp(-((xx[None,None]-center[:,:,0,None,None])**2+
                       (yy[None,None]-center[:,:,1,None,None])**2)/(2*CFG['sigma']**2))
    target=target/target.sum((-2,-1),keepdim=True).clamp_min(1e-9)
    ce=-(target.flatten(2)*logits.float().flatten(2).log_softmax(-1)).sum(-1)
    return (ce*mask).sum()/mask.sum().clamp_min(1)
def decode(logits):
    # Argmax plus local quarter-pixel refinement, vectorized on GPU.
    hm=logits.float(); b,k,h,w=hm.shape; flat=hm.flatten(2).argmax(-1)
    x=flat%w; y=flat//w
    bi=torch.arange(b,device=hm.device)[:,None]; ji=torch.arange(k,device=hm.device)[None,:]
    dx=.25*torch.sign(hm[bi,ji,y,(x+1).clamp(max=w-1)]-hm[bi,ji,y,(x-1).clamp(min=0)])
    dy=.25*torch.sign(hm[bi,ji,(y+1).clamp(max=h-1),x]-hm[bi,ji,(y-1).clamp(min=0),x])
    xf=x.float()+dx*((x>0)&(x<w-1)); yf=y.float()+dy*((y>0)&(y<h-1))
    return (torch.stack((xf,yf),-1)+.5)*4-.5
def original_coordinates(points,rows):
    geo=np.array([crop_geometry(r['bbox']) for r in rows])
    return points*(geo[:,None,2:]/np.array([W,H]))+geo[:,None,:2]
def error_arrays(pred,rows):
    truth=np.array([r['keypoints'] for r in rows]); boxes=np.array([r['bbox'] for r in rows])
    errors=np.linalg.norm(pred-truth[:,:,:2],axis=-1)/np.linalg.norm(boxes[:,2:],axis=-1)[:,None]
    return errors,truth[:,:,2]
def summarize(errors,mask):
    e=errors[mask]
    if not len(e): return dict(count=0,PCK05=np.nan,PCK10=np.nan,NME=np.nan,AUC020=np.nan)
    return dict(count=len(e),PCK05=float((e<=.05).mean()),PCK10=float((e<=.10).mean()),
                NME=float(e.mean()),AUC020=float(np.maximum(1-e/.2,0).mean()))
@torch.no_grad()
def predict(model,dataset):
    loader=DataLoader(dataset,batch_size=CFG['batch_size'],shuffle=False,num_workers=0)
    model.eval(); predictions=[]; losses=[]
    for images,points,mask,indices in loader:
        images,points,mask=images.to(DEVICE),points.to(DEVICE),mask.to(DEVICE)
        logits=model(images); losses.append((heatmap_loss(logits,points,mask).item(),float(mask.sum())))
        predictions.append(decode(logits).cpu().numpy())
    pred=original_coordinates(np.concatenate(predictions),dataset.rows)
    return pred,sum(v*n for v,n in losses)/sum(n for _,n in losses)

## 6. Train, monitor, pause and resume

Each epoch updates `history.csv`, `status.json`, `learning_curves.png`, and the atomic `last.pt` checkpoint. `best.pt` is selected on validation only. Training uses mixed precision and gradient clipping. Checkpoints retain model, optimizer, scheduler, scaler, RNG states and history. A configuration and split fingerprint prevents accidental incompatible resumes.

**Pause:** interrupt this cell or create a `PAUSE` file from another terminal, from the repository root:
```powershell
New-Item ml/runs/cattle_resnet18_p1A/PAUSE -ItemType File -Force
```
**Resume:** remove `PAUSE`, then rerun cells in order. An incomplete epoch is replayed from the last completed checkpoint; at most one epoch is lost. Deterministic per-sample augmentation and epoch shuffling reproduce that epoch. Never run two trainers in this directory. Checkpoints must be your own trusted local files.

Read `history.csv` or open `learning_curves.png` while training. Live batch progress and epoch summaries also appear below. Early stopping uses validation PCK@.05 with 20 epochs of patience.

In [ ]:
def atomic_json(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp'); tmp.write_text(json.dumps(value,indent=2)); tmp.replace(path)
def atomic_save(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp'); torch.save(value,tmp); tmp.replace(path)
def plot_history(history,show=False):
    if not history: return
    frame=pd.DataFrame(history); fig,ax=plt.subplots(1,2,figsize=(11,3.5))
    ax[0].plot(frame.epoch,frame.train_loss,label='train'); ax[0].plot(frame.epoch,frame.val_loss,label='validation')
    ax[0].set(xlabel='Completed epoch',ylabel='Spatial cross entropy'); ax[0].legend()
    ax[1].plot(frame.epoch,frame.PCK05,label='PCK@.05'); ax[1].plot(frame.epoch,frame.PCK10,label='PCK@.10')
    ax[1].set(xlabel='Completed epoch',ylabel='Validation PCK',ylim=(0,1)); ax[1].legend()
    fig.tight_layout(); fig.savefig(RUN/'learning_curves.png',dpi=130)
    if show: plt.show()
    plt.close(fig)
def train():
    if (RUN/'PAUSE').exists():
        print('Paused: remove',RUN/'PAUSE','and rerun.'); return False
    signature=dict(config=CFG,source_commit=COMMIT,split_hash=SPLIT_HASH)
    config_path=RUN/'config.json'
    if config_path.exists(): assert json.loads(config_path.read_text())==signature, 'Configuration changed: use a new run directory.'
    else: atomic_json(config_path,signature)
    last_path=RUN/'last.pt'
    model=CattlePose(pretrained=CFG['pretrained'] and not last_path.exists()).to(DEVICE)
    optimizer=torch.optim.AdamW([{'params':model.backbone.parameters(),'lr':CFG['lr']*.1},
                                 {'params':model.decoder.parameters(),'lr':CFG['lr']}],weight_decay=CFG['weight_decay'])
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=CFG['epochs'],eta_min=1e-6)
    scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')
    history=[]; start=0; best=-1.; bad=0
    def checkpoint(epoch):
        return dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),
                    scaler=scaler.state_dict(),epoch=epoch,best=best,bad=bad,history=history,signature=signature,
                    torch_rng=torch.get_rng_state(),numpy_rng=np.random.get_state(),python_rng=random.getstate(),
                    cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else None)
    if last_path.exists():
        state=torch.load(last_path,map_location='cpu')
        assert state['signature']==signature
        model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']); scaler.load_state_dict(state['scaler'])
        start=state['epoch']+1; best=state['best']; bad=state['bad']; history=state['history']
        torch.set_rng_state(state['torch_rng']); np.random.set_state(state['numpy_rng']); random.setstate(state['python_rng'])
        if DEVICE.type=='cuda' and state['cuda_rng'] is not None: torch.cuda.set_rng_state_all(state['cuda_rng'])
        print(f'Resuming after epoch {start}; best validation PCK05={best:.4f}')
    else: atomic_save(last_path,checkpoint(-1))
    atomic_json(RUN/'environment.json',ENV)
    if history: pd.DataFrame(history).to_csv(RUN/'history.csv',index=False)
    try:
        for epoch in range(start,CFG['epochs']):
            if bad>=CFG['patience']: break
            if (RUN/'PAUSE').exists(): raise KeyboardInterrupt
            started=time.perf_counter(); train_ds.epoch=epoch; model.train(); frozen=epoch<CFG['freeze_epochs']
            for parameter in model.backbone.parameters(): parameter.requires_grad_(not frozen)
            if frozen: model.backbone.eval()
            generator=torch.Generator().manual_seed(CFG['seed']+epoch)
            loader=DataLoader(train_ds,batch_size=CFG['batch_size'],shuffle=True,generator=generator,num_workers=0,pin_memory=DEVICE.type=='cuda')
            total=0.; count=0.
            bar=tqdm(loader,desc=f"Epoch {epoch+1}/{CFG['epochs']}",leave=False)
            for images,points,mask,_ in bar:
                if (RUN/'PAUSE').exists(): raise KeyboardInterrupt
                images,points,mask=images.to(DEVICE),points.to(DEVICE),mask.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'):
                    loss=heatmap_loss(model(images),points,mask)
                assert torch.isfinite(loss), 'Nonfinite training loss'
                scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(),1.)
                scaler.step(optimizer); scaler.update()
                number=float(mask.sum()); total+=loss.item()*number; count+=number
                bar.set_postfix(loss=f'{total/max(count,1):.3f}')
            val_pred,val_loss=predict(model,val_ds)
            errors,visibility=error_arrays(val_pred,records['val']); metrics=summarize(errors,visibility>0)
            score=metrics['PCK05']; improved=score>best+1e-5
            if improved: best=score; bad=0
            else: bad+=1
            scheduler.step()
            row=dict(epoch=epoch+1,train_loss=total/count,val_loss=val_loss,**metrics,
                     seconds=time.perf_counter()-started,lr=optimizer.param_groups[1]['lr'])
            history.append(row); atomic_save(last_path,checkpoint(epoch))
            if improved: atomic_save(RUN/'best.pt',checkpoint(epoch))
            pd.DataFrame(history).to_csv(RUN/'history.csv',index=False)
            atomic_json(RUN/'status.json',dict(state='training',best_val_PCK05=best,patience_used=bad,**row))
            plot_history(history)
            print(f"Epoch {epoch+1:3d} | train {total/count:.3f} | val {val_loss:.3f} | PCK05 {score:.3%} | PCK10 {metrics['PCK10']:.3%} | {row['seconds']:.1f}s",flush=True)
        atomic_json(RUN/'status.json',dict(state='complete',completed_epoch=len(history),best_val_PCK05=best,
                                         reason='early stopping' if bad>=CFG['patience'] else 'epoch budget'))
        plot_history(history,show=True); return True
    except KeyboardInterrupt:
        atomic_json(RUN/'status.json',dict(state='paused',completed_epoch=len(history),best_val_PCK05=best))
        print('Paused safely after completed epoch',len(history),'; incomplete epoch will be replayed.'); return False
TRAINING_COMPLETE=train()

## 7. Held-out evaluation and train-only baseline

Load the checkpoint selected by validation, then evaluate test. Fit a constant mean-shape baseline on training labels normalized to each bounding box. Report all labeled, visible and occluded joints, plus per-joint metrics.

PCK confidence intervals use 1,000 bootstrap resamples of whole test images, preserving joint correlations within an image. They describe sampling variation within this test set, not uncertainty across farms or training seeds. Predictions are cached against the selected checkpoint hash. Never select a checkpoint using these test results.

In [ ]:
assert TRAINING_COMPLETE, 'Training paused: resume before final test evaluation.'
state=torch.load(RUN/'best.pt',map_location='cpu')
model=CattlePose(pretrained=False).to(DEVICE); model.load_state_dict(state['model']); model.eval()
checkpoint_hash=hashlib.sha256((RUN/'best.pt').read_bytes()).hexdigest(); cache_path=RUN/'test_predictions.npz'
if cache_path.exists():
    cached=np.load(cache_path)
    assert str(cached['checkpoint_hash'])==checkpoint_hash, 'Checkpoint changed: use a new evaluation directory.'
    test_pred=cached['predictions']
else:
    test_ds=CattleDataset(records['test']); test_pred,_=predict(model,test_ds)
    np.savez_compressed(cache_path,predictions=test_pred,checkpoint_hash=checkpoint_hash)
errors,visibility=error_arrays(test_pred,records['test'])
train_kp=np.array([r['keypoints'] for r in records['train']]); train_boxes=np.array([r['bbox'] for r in records['train']])
normalized=(train_kp[:,:,:2]-train_boxes[:,None,:2])/train_boxes[:,None,2:]
normalized[train_kp[:,:,2]==0]=np.nan; mean_shape=np.nanmean(normalized,axis=0)
assert np.isfinite(mean_shape).all(), 'A joint has no training labels.'
test_boxes=np.array([r['bbox'] for r in records['test']])
baseline_pred=mean_shape[None]*test_boxes[:,None,2:]+test_boxes[:,None,:2]
baseline_errors,_=error_arrays(baseline_pred,records['test']); summary=[]
for name,e in [('ResNet18 heatmaps',errors),('Train mean shape',baseline_errors)]:
    for subset,mask in [('all labeled',visibility>0),('visible',visibility==2),('occluded',visibility==1)]:
        summary.append(dict(model=name,subset=subset,**summarize(e,mask)))
summary=pd.DataFrame(summary); display(summary.round(4)); summary.to_csv(RUN/'test_metrics.csv',index=False)
per_joint=pd.DataFrame([dict(keypoint=name,**summarize(errors[:,j],visibility[:,j]>0)) for j,name in enumerate(NAMES)])
display(per_joint.round(4)); per_joint.to_csv(RUN/'per_joint_metrics.csv',index=False)
rng=np.random.default_rng(CFG['seed']); boots=[]
for _ in range(1000):
    indices=rng.integers(0,len(errors),len(errors)); boots.append(summarize(errors[indices],visibility[indices]>0)['PCK05'])
ci=np.quantile(boots,[.025,.975])
report=dict(checkpoint_epoch=state['epoch']+1,checkpoint_sha256=checkpoint_hash,validation_PCK05=state['best'],
            test_PCK05=summarize(errors,visibility>0)['PCK05'],test_PCK05_image_bootstrap_CI95=ci.tolist(),
            protocol='p1A with train/val filename repair; ground-truth boxes; bbox-diagonal PCK',
            source=provenance,split_hash=SPLIT_HASH,config=CFG)
atomic_json(RUN/'evaluation.json',report)
print(f"Selected epoch: {state['epoch']+1}; test PCK05 image-bootstrap 95% CI: [{ci[0]:.3%}, {ci[1]:.3%}]")

## 8. Performance curves and qualitative output

Green crosses are labels; red points are predictions. Show a fixed random test sample and the six largest mean errors. The network latency benchmark excludes detection, image loading, cropping, and decoding; it is not end-to-end latency.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(18,5)); thresholds=np.linspace(0,.2,101)
for name,e in [('ResNet18',errors),('Mean shape',baseline_errors)]:
    axes[0].plot(thresholds,[(e[visibility>0]<=t).mean() for t in thresholds],label=name)
axes[0].set(xlabel='Error / original bbox diagonal',ylabel='PCK',ylim=(0,1)); axes[0].legend(); axes[0].grid(alpha=.25)
axes[1].barh(per_joint.keypoint,per_joint.PCK05); axes[1].set(xlabel='Test PCK@.05',xlim=(0,1)); axes[1].invert_yaxis()
for v,label in [(2,'Visible'),(1,'Occluded')]:
    e=errors[visibility==v]
    if len(e): axes[2].hist(e,bins=np.linspace(0,.4,30),alpha=.5,label=label,density=True)
axes[2].set(xlabel='Normalized error (display range 0 to .4)',ylabel='Density'); axes[2].legend()
plt.tight_layout(); plt.savefig(RUN/'test_performance.png',dpi=150); plt.show()
def show_predictions(indices,title,filename):
    fig,axes=plt.subplots(2,3,figsize=(15,8))
    for ax,index in zip(axes.flat,indices):
        row=records['test'][int(index)]; image,kp=make_crop(row); geo=crop_geometry(row['bbox'])
        pred=(test_pred[index]-geo[:2])/geo[2:]*[W,H]; m=kp[:,2]>0
        ax.imshow(image); ax.scatter(kp[m,0],kp[m,1],c='lime',marker='x',s=25,label='Label')
        ax.scatter(pred[m,0],pred[m,1],c='red',s=12,label='Prediction')
        for j in np.flatnonzero(m): ax.plot([kp[j,0],pred[j,0]],[kp[j,1],pred[j,1]],color='yellow',alpha=.5,lw=.6)
        ax.set_title(f"Image {row['image_id']} | NME {errors[index,m].mean():.3f}"); ax.axis('off')
    axes.flat[0].legend(loc='lower left',fontsize=8); fig.suptitle(title); fig.tight_layout()
    fig.savefig(RUN/filename,dpi=140); plt.show()
show_predictions(np.random.default_rng(CFG['seed']).choice(len(errors),6,replace=False),'Fixed random test examples','test_examples.png')
mean_error=np.nanmean(np.where(visibility>0,errors,np.nan),axis=1)
show_predictions(np.argsort(mean_error)[-6:][::-1],'Six largest test errors','test_failures.png')
@torch.no_grad()
def benchmark():
    sample=torch.randn(CFG['batch_size'],3,H,W,device=DEVICE)
    for _ in range(10): model(sample)
    if DEVICE.type=='cuda': torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    timings=[]
    for _ in range(30):
        begin=time.perf_counter(); model(sample)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        timings.append(time.perf_counter()-begin)
    return dict(device=str(DEVICE),batch_size=CFG['batch_size'],crop_network_ms_per_image=1000*np.mean(timings)/CFG['batch_size'],
                crop_network_images_per_second=CFG['batch_size']/np.mean(timings),parameters=sum(p.numel() for p in model.parameters()),
                peak_cuda_MB=torch.cuda.max_memory_allocated()/1024**2 if DEVICE.type=='cuda' else None)
latency=benchmark(); atomic_json(RUN/'benchmark.json',latency); display(pd.DataFrame([latency]))

## 9. Inference on a new image and supplied cattle box

Return original-image coordinates in the documented joint order. No keypoint labels are needed. This model does not infer visibility, recognize cattle identities, or diagnose health. Accuracy here does not establish generalization to drone views.

Further work: independently identified farms/videos/animals, multiple training seeds, untouched source-group folds, and detector-plus-pose evaluation. Tune on validation only; the p1A test set has been consulted after this run.

In [ ]:
@torch.no_grad()
def infer_cattle(image_path,bbox):
    path=Path(image_path)
    with Image.open(path) as im: iw,ih=im.size
    x,y,w,h=map(float,bbox)
    assert x>=0 and y>=0 and w>0 and h>0 and x+w<=iw and y+h<=ih
    row=dict(path=str(path),bbox=[x,y,w,h],keypoints=np.zeros((K,3)).tolist()); crop,_=make_crop(row)
    tensor=torch.from_numpy(((crop.astype(np.float32)/255-MEAN)/STD).transpose(2,0,1).copy())[None].to(DEVICE)
    prediction=original_coordinates(decode(model(tensor)).cpu().numpy(),[row])[0]
    return pd.DataFrame({'keypoint':NAMES,'x':prediction[:,0],'y':prediction[:,1]})
# infer_cattle('path/to/cow.jpg', [100, 120, 600, 420])
print('Artifacts:',RUN)